# 01 — Funções auxiliares

Utilitários reutilizáveis para metadados, validação, conversões de cor/tipo, máscaras e operações pontuais na ROI.

**Dependências:** `numpy`.

**Ordem recomendada:** executar este notebook antes de `06-uploading` (pré-processamento no carregamento) e dos pipelines de segmentação.


## Importações


In [ ]:
import numpy as np


## Metadados e validação dimensional

Inspeção de propriedades da imagem e verificação de compatibilidade com máscaras.


In [ ]:
def analisar_metadados(img):
    """Devolve shape, dtype e número de canais."""
    canais = 1 if img.ndim == 2 else img.shape[2]
    return {"shape": img.shape, "dtype": str(img.dtype), "canais": canais}


def validar_dimensoes(img, mask=None):
    """True se imagem e máscara têm as mesmas dimensões espaciais (ou mask é None)."""
    if mask is None:
        return True
    return img.shape[:2] == mask.shape[:2]


## Conversão de tipo e cor

Normalização `uint8`, grayscale (BT.601) e binarização por intervalo.


In [ ]:
def garantir_uint8(img):
    """Converte para uint8 em [0, 255] com normalização min-max."""
    if img.dtype == np.uint8:
        return img
    img_min = float(np.min(img))
    img_max = float(np.max(img))
    if img_max == img_min:
        return np.zeros_like(img, dtype=np.uint8)
    escalada = (img - img_min) / (img_max - img_min) * 255.0
    return escalada.astype(np.uint8)


def converter_para_grayscale(img):
    """RGB/RGBA → grayscale com pesos 0.299, 0.587, 0.114."""
    if img.ndim == 2:
        return img
    rgb = img[:, :, :3] if img.shape[2] == 4 else img
    return (
        0.299 * rgb[:, :, 0]
        + 0.587 * rgb[:, :, 1]
        + 0.114 * rgb[:, :, 2]
    )


def binarizar_intervalo(img_gray, intervalo_min, intervalo_max):
    """Pixels em [min, max] → 255; restantes → 0 (uint8)."""
    binaria = (img_gray >= intervalo_min) & (img_gray <= intervalo_max)
    return binaria.astype(np.uint8) * 255


def converter_para_grayscale_e_binaria(img, intervalo=None):
    """Grayscale; com intervalo=(min, max) devolve máscara binária uint8."""
    gray = converter_para_grayscale(img)
    if intervalo is None:
        return gray
    return binarizar_intervalo(gray, intervalo[0], intervalo[1])


## Validação de máscaras


In [ ]:
def validar_mascara_binaria(mask, tolerancia=0.02):
    """Valida valores {0, 255}; corrige por limiar 127 se dentro da tolerância."""
    total = mask.size
    nao_binarios = np.sum((mask != 0) & (mask != 255))
    if nao_binarios / total > tolerancia:
        return False, mask
    corrigida = np.where(mask < 127, 0, 255).astype(np.uint8)
    return True, corrigida


def verificar_fundo(img, mask, limiar_escuro=30, tolerancia=0.02):
    """Verifica coerência entre fundo da máscara (0) e regiões escuras na imagem."""
    fundo = mask == 0
    total_fundo = int(np.sum(fundo))
    if total_fundo == 0:
        return False
    incoerencias = int(np.sum((img >= limiar_escuro) & fundo))
    return (incoerencias / total_fundo) <= tolerancia


## ROI e histograma


In [ ]:
def extrair_roi(img, mask, valor_mascara=255):
    """Extrai vetor de pixels e imagem parcial na ROI."""
    roi_pixels = img[mask == valor_mascara]
    roi_img = np.zeros_like(img)
    roi_img[mask == valor_mascara] = img[mask == valor_mascara]
    return roi_pixels, roi_img


def calcular_histograma_roi(img, mask, valor_mascara=255):
    """Histograma 256 níveis apenas na ROI."""
    hist = np.zeros(256, dtype=int)
    roi = img[mask == valor_mascara].ravel().astype(int)
    for intensidade in roi:
        if 0 <= intensidade <= 255:
            hist[intensidade] += 1
    return hist, np.arange(257)


## Transformações pontuais na ROI


In [ ]:
def aplicar_contraste_brilho_roi(img, mask, alpha=1.0, beta=0, valor_mascara=255):
    """g = alpha*f + beta na ROI; valores limitados a [0, 255]."""
    saida = img.copy().astype(np.float64)
    roi = mask == valor_mascara
    saida[roi] = np.clip(alpha * saida[roi] + beta, 0, 255)
    return saida.astype(np.uint8)


def inverter_roi(img, mask, valor_mascara=255):
    """Inversão g = 255 - f apenas na ROI."""
    saida = img.copy()
    roi = mask == valor_mascara
    saida[roi] = 255 - saida[roi]
    return saida
